In [1]:
import xarray as xr
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path
import rioxarray
from rasterio.enums import Resampling
import zarr
import shutil
import dask.array as da
import dask
import shutil
import os


root = Path.cwd()


In [ ]:

def cloudmask_and_save(siteyear,input,filename,epsg,clip_raster=False):

    if clip_raster:
        b = gpd.read_file(root / 'data' / 'nyc_boundary.gpkg')
        ref_10m = input.rio.clip(b.geometry)
        scl_values = [1, 3, 6, 8, 9, 10, 11]
    else:
        ref_10m = input
        scl_values = [1, 3, 8, 9, 10, 11] ### don't mask out water (6) yet because of glcm edge effects

    # cloud mask
    scl_band = xr.open_zarr(root / 'data' / siteyear / f'{siteyear}_daily_scl_10m.zarr')
    scl_band = scl_band[[i for i in scl_band.data_vars][0]]
    scl_band = scl_band.rio.write_crs(epsg).rio.set_spatial_dims(x_dim='x',y_dim='y').rio.write_coordinate_system()

    ## TODO: assert alignment

    scl_band = scl_band.sel(time=ref_10m.time.values,band='SCL')
    
    mask = ~scl_band.isin(scl_values)

    ref_10m_masked = ref_10m.where(mask)

    # compress and save
    bandchunksize = len(ref_10m_masked.band.values)

    ref_10m_masked.name = 'refl'
    encoding = {'refl':{'dtype':'float32',
                        'zlib':True,
                        'complevel':4,
                        'shuffle':True,
                        'chunksizes':(1,bandchunksize,512,512)}}

    ref_10m_masked.to_netcdf(root / 'data' / siteyear / f'{filename}.nc',engine='netcdf4',encoding=encoding)





#### Cloud mask native 10m bands

In [5]:
## TODO: add to initial download function

# read in native 10m data
siteyear = 'nyc17'
data = xr.open_dataarray(root / 'data' / siteyear / f'{siteyear}_daily_10m_aligned_buffered.nc')
ref_10m = data.rio.write_crs(26918).rio.set_spatial_dims(x_dim='x',y_dim='y').rio.write_coordinate_system()

cloudmask_and_save(siteyear=siteyear,input=ref_10m,epsg=26918,clip_raster=False,filename=f'{siteyear}_bands2348_ARD_buffered')

#### Cloud mask DSen2 numpy outputs

In [ ]:
# read in downsampled data
npy_list = sorted(os.listdir(root / 'data' / 'nyc21' / 'dsen2_output'))

files = [np.load(root / 'data' / 'nyc21' / 'dsen2_output' / n ) for n in npy_list]
npy_stack = np.stack(files,axis=0)
npy_stack = npy_stack.transpose(0,3,1,2)

ref_ds = xr.DataArray(npy_stack,dims=('time','band','y','x'),coords={'time':ref_10m.time,'band':['B05','B06','B07','B8A','B11','B12'],'y':ref_10m.y,'x':ref_10m.x})
ref_ds = ref_ds.rio.write_crs(26918).rio.set_spatial_dims(x_dim='x',y_dim='y').rio.write_coordinate_system()


cloudmask_and_save(input=ref_ds,filename='nyc21_bands5to11_downsampled10m_ARD')

In [ ]:
## lazy version
npy_list = sorted(os.listdir(root / 'data' / 'nyc21' / 'dsen2_output'))

files = [da.from_delayed(
            dask.delayed(np.load)(root / 'data' / 'nyc21' / 'dsen2_output' / n),
            shape=ref_10m.shape,
            dtype='float32'
         ) for n in npy_list]

npy_stack = da.stack(files, axis=0).transpose(0, 3, 1, 2).rechunk((1, -1, 512, 512))

ref_ds = xr.DataArray(npy_stack, dims=('time','band','y','x'),
                      coords={'time': ref_10m.time,
                              'band': ['B05','B06','B07','B8A','B11','B12'],
                              'y': ref_10m.y,
                              'x': ref_10m.x})

cloudmask_and_save(input=ref_ds,filename='nyc21_bands5to11_downsampled10m_ARD')

#### Resample 20m bands and cloud mask

In [6]:


## resample 20m to 10m using bilinear resampling

## read in 20m data
data = xr.open_dataarray(root / 'data' / siteyear / f'{siteyear}_daily_20m_aligned_buffered.nc')
ref_20m = data.rio.write_crs(26918).rio.set_spatial_dims(x_dim='x',y_dim='y').rio.write_coordinate_system()

template = ref_10m.isel(time=0)

out_path = root / 'data' / siteyear / f'{siteyear}_bands5to11_resampled10m_unmasked.zarr'
if os.path.exists(out_path):
    shutil.rmtree(out_path)

for i, t in enumerate(ref_20m.time.values):
    d = ref_20m.sel(time=t)
    d_resampled = d.rio.reproject_match(template,resampling=Resampling.bilinear).astype('float32').expand_dims({'time':[t]}).transpose('time','band','y','x').assign_coords(time=[np.datetime64(t,'ns')])
    
    ds = d_resampled.to_dataset(name='20m_to_10m_bilinear')

    if i == 0:
        
        encoding = {'20m_to_10m_bilinear':{'chunks':(1,d.sizes['band'],512,512),
                                           'compressor': zarr.Blosc(cname='zstd',clevel=3,shuffle=2)}}

        ds.to_zarr(out_path,mode='w',encoding=encoding)
    
    else:
        ds.to_zarr(out_path,mode='a',append_dim='time')
    
    print(f'completed {t}')



completed 2017-03-17T00:00:00.000000000
completed 2017-04-16T00:00:00.000000000
completed 2017-04-23T00:00:00.000000000
completed 2017-05-16T00:00:00.000000000
completed 2017-06-02T00:00:00.000000000
completed 2017-06-12T00:00:00.000000000
completed 2017-06-15T00:00:00.000000000
completed 2017-06-22T00:00:00.000000000
completed 2017-06-25T00:00:00.000000000
completed 2017-07-02T00:00:00.000000000
completed 2017-07-05T00:00:00.000000000
completed 2017-07-30T00:00:00.000000000
completed 2017-08-01T00:00:00.000000000
completed 2017-08-04T00:00:00.000000000
completed 2017-08-16T00:00:00.000000000
completed 2017-08-19T00:00:00.000000000
completed 2017-08-21T00:00:00.000000000
completed 2017-08-24T00:00:00.000000000
completed 2017-08-26T00:00:00.000000000
completed 2017-08-31T00:00:00.000000000
completed 2017-09-05T00:00:00.000000000
completed 2017-09-08T00:00:00.000000000
completed 2017-09-10T00:00:00.000000000
completed 2017-09-13T00:00:00.000000000
completed 2017-09-15T00:00:00.000000000


In [7]:
### read in resampled data and apply cloud mask
data = xr.open_zarr(root / 'data' / siteyear / f'{siteyear}_bands5to11_resampled10m_unmasked.zarr')['20m_to_10m_bilinear']

cloudmask_and_save(siteyear=siteyear,input=data,clip_raster=False,filename=f'{siteyear}_bands5to11_resampled10m_ARD',epsg=26918)

### Derive spectral indices

In [ ]:
from dask.distributed import Client as da_client
from dask.distributed import LocalCluster

cluster = LocalCluster(
    n_workers=4,              
    threads_per_worker=1,     
    memory_limit='6GB',       
)
dask_client = da_client(cluster)
dask_client

In [3]:
def derive_indices(ref_10m,boundary,epsg):

    ref = ref_10m.rio.write_crs(epsg).rio.set_spatial_dims(x_dim='x',y_dim='y').rio.write_coordinate_system()
    ref = ref.rio.clip(boundary.geometry)
   
    blue  = ref.sel(band="B02")
    green = ref.sel(band="B03")
    red   = ref.sel(band="B04")
    re1   = ref.sel(band="B05")
    re2   = ref.sel(band="B06")
    re3   = ref.sel(band="B07")
    nir   = ref.sel(band="B08")
    sw1   = ref.sel(band="B11")
    sw2   = ref.sel(band="B12")

    ds = xr.Dataset()

    eps = 1e-10

    ds["tdvi"]  = 1.5 * ((nir - red) / (nir**2 + red + 0.5) ** 0.5)
    ds["s2wi"]  = (re1 - sw2) / (re1 + sw2 + eps)
    ds["ndmi"]  = (nir - sw1) / (nir + sw1 + eps)
    ds["ndre2"] = (nir - re2) / (nir + re2 + eps)
    ds["ireci"] = (re3 - red) / (re1 / (re2 +eps) + eps)
    ds["psri"]  = (red - blue) / (re2 + eps)
    ds["nbr"]   = (nir - sw2) / (nir + sw2 + eps)
    ds["msi"]   = sw1 / (nir + eps)
    ds["evi"]   = 2.5 * (nir - red) / (nir + 6*red - 7.5*blue + 1.0)
    ds["gndvi"] = (nir - green) / (nir + green + eps)
    ds["grvi"]  = (green - red) / (green + red + eps)
    ds["ndbi"]  = (sw1 - nir) / (sw1 + nir + eps)
    ds["bsi"] = ((sw1 + red) - (nir + blue)) / ((sw1 + red) + (nir + blue) + eps)

    all_indices = ds.to_array(dim='band').transpose('time', 'band', 'y', 'x')

    all_data = xr.concat([ref,all_indices],dim='band').astype('float32').chunk({'time':-1,'band':5,'y':512,'x':512})
    all_data.name = 'variables'
    encoding = {'variables':{'chunks':(-1,5,512,512),'compressor': zarr.Blosc(cname='zstd',clevel=3,shuffle=2)}}
    all_data.to_zarr(root / 'data' / siteyear / f'{siteyear}_allbandsandindices.zarr',mode='w',encoding=encoding)
 

In [4]:
siteyear = 'nyc18'
ref = xr.open_zarr(root / 'data' / siteyear / f'{siteyear}_daily_raw_10m_aligned.zarr')['reflectance']
ref = ref.chunk(chunks={'time': 4, 'band': -1, 'y': 512, 'x': 512})

boundary = gpd.read_file(root / 'data' / 'nyc_boundary.gpkg')


In [5]:
derive_indices(ref_10m=ref,boundary=boundary,epsg=26918)

c:\Users\roseh\miniconda3\envs\utc2-env\Lib\site-packages\distributed\client.py:3371: UserWarning: Sending large graph of size 24.38 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


#### Derive Seasonal Statistics

In [2]:

siteyear = 'nyc18'
sc = 256
all_data = xr.open_zarr(root / 'data' / siteyear / f'{siteyear}_allbandsandindices.zarr')['variables']
all_data = all_data.chunk({'time':-1,'band':5,'y':sc,'x':sc})
bytes = all_data.chunks[0][0] * all_data.chunks[1][0] *  all_data.chunks[2][0] *  all_data.chunks[3][0] *  4
        

mb = bytes/1e6

print(f'MB per chunk: {mb}')


MB per chunk: 52.4288


In [3]:
from dask.distributed import Client as da_client
from dask.distributed import LocalCluster

cluster = LocalCluster(
    n_workers=4,              
    threads_per_worker=1,     
    memory_limit='6GB',       
)
dask_client = da_client(cluster)
dask_client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 4,Total memory: 22.35 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:49975,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:49992,Total threads: 1
Dashboard: http://127.0.0.1:49993/status,Memory: 5.59 GiB
Nanny: tcp://127.0.0.1:49978,


In [4]:
out_path = root / 'data' / siteyear / f'{siteyear}_seasonalstats.zarr'
if os.path.exists(out_path):
    shutil.rmtree(out_path)

all_data = xr.open_zarr(root / 'data' / siteyear / f'{siteyear}_allbandsandindices.zarr')['variables']
all_data = all_data.chunk({'time':-1,'band':5,'y':256,'x':256})

# all keys have same number of letters - necessary for zarr encoding
windows = {"season": (3, 11),"spring": (3, 5),"summer": (6, 8),"autumn": (9, 11)}

out_segments = []
for i, (wname, (m1,m2)) in enumerate(windows.items()):
    print(wname)
    season = all_data.sel(time=all_data.time.dt.month.isin(range(m1,m2+1)))


    # aggregations
    mean = season.mean(dim='time',skipna=True).reset_coords(drop=True)
    mx = season.max(dim='time',skipna=True).reset_coords(drop=True)
    mn = season.min(dim='time',skipna=True).reset_coords(drop=True)
    std = season.std(dim='time',skipna=True).reset_coords(drop=True)

    amp = mx - mn
    cv = std / mean.clip(min=1e-6)

    q = season.quantile([0.25, 0.5, 0.75], dim="time", skipna=True)
    p25 = q.sel(quantile=0.25).drop_vars("quantile")
    p50 = q.sel(quantile=0.5 ).drop_vars("quantile")
    p75 = q.sel(quantile=0.75).drop_vars("quantile")
    iqr = p75 - p25

    stats = {'mean':mean,
             'max':mx,
             'min':mn,
             'std':std,
             'amp':amp,
             'cv':cv,
             'p25':p25,
             'p50':p50,
             'p75':p75,
             'iqr':iqr}
    
    ## add new band coordinate to each stat array
    segments = []
    for st in stats:
        a = stats[st]
        new_band = a['band'].astype('str') + f'_{wname}_{st}'
        a = a.assign_coords({'band':new_band})
        segments.append(a)
    
    all_segs = xr.concat(segments,dim='band')
    all_segs = all_segs.to_dataset(name='features').astype('float32')

    band_chunk = all_segs.sizes['band']
    spatial_chunk = all_data.chunks[2][0]

    #TODO: convert band coord to object dtype

    ## dask chunks have to match zarr encoding
    all_segs = all_segs.chunk({'band':band_chunk,'y':spatial_chunk,'x':spatial_chunk})
    
    if i == 0:
       
        encoding = {'features':{'chunks':(band_chunk,spatial_chunk,spatial_chunk),
                                           'compressor': zarr.Blosc(cname='zstd',clevel=3,shuffle=2)}}
                                            # compressor works for zarr v2 (not v3)
        
        all_segs.to_zarr(out_path,mode='w',encoding=encoding)
    
    else:
        all_segs.to_zarr(out_path,mode='a',append_dim='band')






    

season
spring
summer
autumn


In [ ]:
import warnings
import numpy as np

with warnings.catch_warnings():
    warnings.filterwarnings('ignore', category=RuntimeWarning, message='invalid value encountered in divide')
    with ProgressBar():

In [7]:
chunks={'time':-1,'band':-1,'y':512,'x':512}

#band_method = 'dsen2'
band_method = 'resampled'
siteyear = 'nyc17'

d1 = xr.open_dataarray(root / 'data' / siteyear / f'{siteyear}_bands2348_ARD_buffered.nc',chunks=chunks)
#d2 = xr.open_dataarray(root / 'data' / 'nyc21' / 'nyc21_bands5to11_dsen210m_ARD.nc',chunks=chunks)
d2 = xr.open_dataarray(root / 'data' / siteyear / f'{siteyear}_bands5to11_{band_method}10m_ARD.nc',chunks=chunks)



all_bands = xr.concat([d1,d2],dim='band')

### clip to boundary and apply water mask
b = gpd.read_file(root / 'data' / 'nyc_boundary.gpkg')
all_bands_clipped = all_bands.rio.clip(b.geometry)

# water mask 
scl_band = xr.open_zarr(root / 'data' / siteyear / f'{siteyear}_daily_scl_10m.zarr')
scl_band = scl_band[[i for i in scl_band.data_vars][0]]
scl_band = scl_band.rio.write_crs(26918).rio.set_spatial_dims(x_dim='x',y_dim='y').rio.write_coordinate_system()

scl_band = scl_band.sel(time=all_bands_clipped.time.values,band='SCL')
    
mask = ~scl_band.isin([6])

all_bands_clipped = all_bands_clipped.where(mask)
all_bands_clipped = all_bands_clipped.chunk(chunks=chunks)

# calculate spectral indices and save as dataset
idx_ds = derive_indices(all_bands_clipped)
# TODO: add compression
idx_ds.to_zarr(root / 'data' / siteyear / f'{siteyear}_indices_dataset_{band_method}.zarr', mode='w')

c:\Users\roseh\miniconda3\envs\utc2-env\Lib\site-packages\distributed\client.py:3371: UserWarning: Sending large graph of size 23.01 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


#### Combine bands and indices into one file

In [8]:

chunks={'time':-1,'band':-1,'y':512,'x':512}

# d1 = xr.open_dataarray(root / 'data' / siteyear / f'{siteyear}_bands2348_ARD_buffered.nc',chunks=chunks)
# d2 = xr.open_dataarray(root / 'data' / siteyear / f'{siteyear}_bands5to11_{band_method}10m_ARD.nc',chunks=chunks)
# all_bands = xr.concat([d1,d2],dim='band')

all_indices = xr.open_zarr(root / 'data' / siteyear / f'{siteyear}_indices_dataset_{band_method}.zarr',chunks=chunks).to_array(dim='band')

all_indices = all_indices.transpose('time', 'band', 'y', 'x')

all_data = xr.concat([all_bands_clipped,all_indices],dim='band').astype('float32').chunk({'time':-1,'band':5,'y':512,'x':512})

encoding = {'refl':{'chunks':(-1,5,512,512),'compressor': zarr.Blosc(cname='zstd',clevel=3,shuffle=2)}}
all_data.to_zarr(root / 'data' / siteyear / f'{siteyear}_allbandsandindices_{band_method}.zarr',mode='w',encoding=encoding)

c:\Users\roseh\miniconda3\envs\utc2-env\Lib\site-packages\distributed\client.py:3371: UserWarning: Sending large graph of size 23.07 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


In [11]:
all_data.name

'refl'

In [3]:
from dask.distributed import Client as da_client
from dask.distributed import LocalCluster

cluster = LocalCluster(
    n_workers=6,              
    threads_per_worker=1,     
    memory_limit='4GB',       
)
dask_client = da_client(cluster)
dask_client

c:\Users\roseh\miniconda3\envs\utc2-env\Lib\site-packages\distributed\node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 55045 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:55045/status,
Dashboard: http://127.0.0.1:55045/status,Workers: 6
Total threads: 6,Total memory: 22.35 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:55046,Workers: 0
Dashboard: http://127.0.0.1:55045/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:55067,Total threads: 1
Dashboard: http://127.0.0.1:55069/status,Memory: 3.73 GiB
Nanny: tcp://127.0.0.1:55049,


In [9]:
out_path = root / 'data' / siteyear / f'{siteyear}_seasonalstats_{band_method}.zarr'
if os.path.exists(out_path):
    shutil.rmtree(out_path)

ds = xr.open_zarr(root / 'data' / siteyear / f'{siteyear}_allbandsandindices_{band_method}.zarr')
all_data = ds[list(ds.data_vars)[0]]

# all keys have same number of letters - necessary for zarr encoding
windows = {"season": (3, 11),"spring": (3, 5),"summer": (6, 8),"autumn": (9, 11)}

out_segments = []
for i, (wname, (m1,m2)) in enumerate(windows.items()):
    print(wname)
    season = all_data.sel(time=all_data.time.dt.month.isin(range(m1,m2+1)))


    # aggregations
    mean = season.mean(dim='time',skipna=True).reset_coords(drop=True)
    mx = season.max(dim='time',skipna=True).reset_coords(drop=True)
    mn = season.min(dim='time',skipna=True).reset_coords(drop=True)
    std = season.std(dim='time',skipna=True).reset_coords(drop=True)

    amp = mx - mn
    cv = std / mean.clip(min=1e-6)

    q = season.quantile([0.25, 0.5, 0.75], dim="time", skipna=True)
    p25 = q.sel(quantile=0.25).drop_vars("quantile")
    p50 = q.sel(quantile=0.5 ).drop_vars("quantile")
    p75 = q.sel(quantile=0.75).drop_vars("quantile")
    iqr = p75 - p25

    stats = {'mean':mean,
             'max':mx,
             'min':mn,
             'std':std,
             'amp':amp,
             'cv':cv,
             'p25':p25,
             'p50':p50,
             'p75':p75,
             'iqr':iqr}
    
    ## add new band coordinate to each stat array
    segments = []
    for st in stats:
        a = stats[st]
        new_band = a['band'].astype('str') + f'_{wname}_{st}'
        a = a.assign_coords({'band':new_band})
        segments.append(a)
    
    all_segs = xr.concat(segments,dim='band')
    all_segs = all_segs.to_dataset(name='features').astype('float32')

    band_chunk = all_segs.sizes['band']
    spatial_chunk = all_data.chunks[2][0]

    #TODO: convert band coord to object dtype

    ## dask chunks have to match zarr encoding
    all_segs = all_segs.chunk({'band':band_chunk,'y':spatial_chunk,'x':spatial_chunk})
    
    if i == 0:
       
        encoding = {'features':{'chunks':(band_chunk,spatial_chunk,spatial_chunk),
                                           'compressor': zarr.Blosc(cname='zstd',clevel=3,shuffle=2)}}
                                            # compressor works for zarr v2 (not v3)

        all_segs.to_zarr(out_path,mode='w',encoding=encoding)
    
    else:
        all_segs.to_zarr(out_path,mode='a',append_dim='band')






    

season
spring
summer
autumn


In [ ]:
### for nyc21 i had to append the existing glcm metrics to the end of the new resampled seasonal stats file (glcm are derived from tdvi which uses only 10m bands.)

# get existing glcm bands
stats = xr.open_zarr(root / 'data' / siteyear / f'{siteyear}_seasonalstats_dsen2.zarr')
glcm = stats.sel(band=stats.band.values[920:])


# get encoding from zarr store
store_path = root / 'data' / siteyear / f'{siteyear}_seasonalstats_{band_method}.zarr'
store = zarr.open(store_path)
store_chunks = store['features'].chunks
band_chunk = store_chunks[0]
spatial_chunk = store_chunks[1]

resampled = xr.open_zarr(store_path)
combined = xr.concat([resampled,glcm], dim='band')

## dask chunks have to match zarr encoding
combined_rechunked = combined.chunk({'band':band_chunk,'y':spatial_chunk,'x':spatial_chunk})

out_path = root / 'data' / siteyear / f'{siteyear}_seasonalstatswithglcm_resampled.zarr'
combined_rechunked.to_zarr(out_path,mode='w')